# Glassdoor web scraper for job reviews
This code was developed by Camilo Pinzón Castellanos as a part of his PhD project at UNED during January 2023. It was inspired by the reference code that can be found in: https://bulletbyte.weebly.com/tech/how-to-scrape-a-companys-glassdoor-reviews-using-python

## 1. Import libraries and functions

First, the libraries will be imported and configured if needed.

In [1]:
## Import libraries

# Import os and time
import os
import time

# Data science related
import numpy as np
import pandas as pd
import math

# String related
import re

# Web scraping related
from bs4 import BeautifulSoup
from urllib.request import Request, urlopen

Now, the functions which will be used through the code will be defined. There are two functions:
 - **review_scrap**: it takes a url and it outputs a pandas dataframe with the reviews scraped
 - **get_reviews**: it takes in the main url of the company, and outputs a pandas dataframe with the different things extracted
 - **get_xxxx**: different functions that will be used inside list comprehensions
 - **get_substars**: This function takes in a tag, and returns the substars
 - **substar_number**: This function will take in the text which indicates the substars, and will transform it to a number

In [2]:
def review_scrap(url):
    
    """
    This function will take in an url, and will output a pandas dataframe where each line is a review, and each column is a
    feature of the review. Missing values will be filled in with -
    """
    
    ## Scrap the web page url content

    # State user agent
    hdr = {'User-Agent': 'Mozilla/5.0'}
    
    # Request ulr
    req = Request(url,headers=hdr)
    
    # Get page
    page = urlopen(req)
    
    # Produce soup from BeautifulSoup
    soup = BeautifulSoup(page, "html.parser")
    
    # Get reviews inside soup
    reviews_in_page = soup.find_all('li', {'class': 'noBorder empReview cf pb-0 mb-0'})
    
    
    ## Get information
    
    # Get the review id
    review_id = [get_review_id(tag) for tag in reviews_in_page]
    
    # Get the summary
    summary = [get_summary(tag) for tag in reviews_in_page]
    
    # Get the date
    date = [get_date(tag) for tag in reviews_in_page]
    
    # Get the job title
    job_title = [get_job_title(tag) for tag in reviews_in_page]
    
    # Get the location
    location = [get_location(tag) for tag in reviews_in_page]
    
    # Get the rating
    overall_rating = [get_rating(tag) for tag in reviews_in_page]
    
    # Get the pros
    pros = [get_pros(tag) for tag in reviews_in_page]
    
    # Get the cons
    cons = [get_cons(tag) for tag in reviews_in_page]
    
    # Get wl_balance
    wl_balance = [get_substars(tag)['Work/Life Balance'] for tag in reviews_in_page]

    # Get culture_values
    culture_values = [get_substars(tag)['Culture & Values'] for tag in reviews_in_page]

    # Get diversity_inclusion
    diversity_inclusion = [get_substars(tag)['Diversity & Inclusion'] for tag in reviews_in_page]

    # Get career_opportunities
    career_opportunities = [get_substars(tag)['Career Opportunities'] for tag in reviews_in_page]

    # Get compensation_benefits
    compensation_benefits = [get_substars(tag)['Compensation and Benefits'] for tag in reviews_in_page]

    # Get senior_management
    senior_management = [get_substars(tag)['Senior Management'] for tag in reviews_in_page]
    
    return pd.DataFrame({'review_id': review_id, 'summary': summary, 'date': date, 'job_title': job_title, 'overall_rating': overall_rating, 'pros': pros, 'cons': cons, 'author_location': location, 'wl_balance': wl_balance, 'culture_values': culture_values, 'diversity_inclusion': diversity_inclusion, 'career_opportunities': career_opportunities, 'compensation_benefits': compensation_benefits, 'senior_management': senior_management})

In [3]:
def get_reviews(url_main, first_pag=1, last_pag=101):
    
    """
    This function takes in two arguments:
     - url_main: it's the ulr of the first page of reviews, without the .htm
     - firts_pag: the number of the first page of reviews
     - last_pag: the number of the last page of reviews
     
     It outputs a pandas dataframe with the different things extracted
    """
    
    # Correct last_pag
    last_pag +=1
    
    # Initialise dataframe
    df = pd.DataFrame()
    
    # Iterate through different pages
    for page_num in range(first_pag, last_pag):
        
        # Create dataframe
        df = pd.concat([df, review_scrap(url_main + "_P" + str(page_num) + ".htm")], ignore_index=True)
        
        # Wait for 1 secs between each iteration
        time.sleep(1)
    
    return df

In [4]:
"""
These functions are coded to be used in list comprehensions. All they do is that, if the .find method returns an error, they
will return a -
"""

def get_review_id(current_tag):
    try:
        review_id_current = current_tag['id']
    except:
        review_id_current = '-'
    return review_id_current

def get_summary(current_tag):
    try:
        summary_current = current_tag.find('a', {'class':'reviewLink'}).text
    except:
        summary_current = '-'
    return summary_current

def get_date(current_tag):
    try:
        date_current = current_tag.find('span', {'class': 'authorInfo'}).contents[0].text.split(' - ')[0]
    except:
        try:
            date_current = current_tag.find('span', {'class': 'common__EiReviewDetailsStyle__newUiJobLine'}).contents[0].text.split(' - ')[0]
        except:
            date_current = '-'
    return date_current

def get_job_title(current_tag):
    try:
        job_title_current = current_tag.find('span', {'class': 'authorInfo'}).contents[0].text.split(' - ')[1]
    except:
        try:
            job_title_current = current_tag.find('span', {'class': 'common__EiReviewDetailsStyle__newUiJobLine'}).contents[0].text.split(' - ')[1].split('\xa0in ')[0]
        except:
            job_title_current = '-'
    return job_title_current

def get_location(current_tag):
    try:
        location_current = current_tag.find('span', {'class':'authorLocation'}).text
    except:
        try:
            location_current = current_tag.find('span', {'class': 'common__EiReviewDetailsStyle__newUiJobLine'}).contents[0].text.split(' - ')[1].split('\xa0in ')[1]
        except:
            location_current = '-'
    return location_current

def get_rating(current_tag):
    try:
        rating_current = current_tag.find('span', {'class':'ratingNumber mr-xsm'}).text
    except:
        rating_current = '-'
    return rating_current

def get_pros(current_tag):
    try:
        pros_current = current_tag.find('span', {'data-test':'pros'}).text
    except:
        pros_current = '-'
    return pros_current

def get_cons(current_tag):
    try:
        cons_current = current_tag.find('span', {'data-test':'cons'}).text
    except:
        cons_current = '-'
    return cons_current

In [5]:
def get_substars(current_tag):
    
    """
    This function takes in a tag, and returns the substars
    """
    
    # State list with all substars
    subcategories = ['Work/Life Balance', 'Culture & Values', 'Diversity & Inclusion', 'Career Opportunities',
            'Compensation and Benefits', 'Senior Management']
    
    # Initiate dictionary with all '-'
    substar_info = {key: '-' for key in subcategories}
    
    # Check if review has substars    
    if not(len(current_tag.find_all('aside', {'class': 'gd-ui-tooltip-info toolTip tooltip css-15sm7sw e1mbtzkk0'})) == 1):
        
        # If it doesn't have substars: fill all categories with
        return substar_info
        
    # Get all the info about substars
    substar_info_soup = current_tag.find('div', {'class': 'tooltipContainer'}).contents[1].find_all('div')
    
    # Iterate through different 
    for i in range(0, len(substar_info_soup), 2):
        
        # Save current element as key, and following element as stars
        substar_info[substar_info_soup[i].text] = substars_number(substar_info_soup[i+1]['class'][0]) 
    
    return substar_info

In [6]:
def substars_number(div_text):
    """
    This function will take in the text which indicates the substars, and will transform it to a number
    """
    
    if div_text == 'css-1mfncox': return '1'
    elif div_text == 'css-1lp3h8x': return '2'
    elif div_text == 'css-k58126': return '3'
    elif div_text == 'css-94nhxw': return '4'
    elif div_text == 'css-11w4osi': return '5'
    else: return '-'

## 2. Getting companies
The companies whose information will be obtained are S&P500 companies. For this, a file will be read with the name of each of the companies. This was obtained from: https://github.com/datasets/s-and-p-500-companies/blob/master/data/constituents.csv

Take into account that the last column (the URL for glassdoor) was added manually.

In [7]:
## Read file to pandas

# Actually read file
company_info = pd.read_csv("./sp500_companies.csv")

----------------------------------

Now, we will iterate through the different companies obtaining their reviews.

In [8]:
for company, gd_link in zip(company_info['Symbol'], company_info['Glassdoor link']):
    
    # Check whether we already have reviews for the company
    if company + "_reviews.csv" in os.listdir("./01_company_reviews"):
        
        # Go to next company
        continue
    
    
    # Print current company for reference
    print("\nThe current company is: " + company)
    
    
    # Try with first 25 pages
    try: # 2-25 reviews
        
        # Print state
        print("   Getting reviews 2-25")
        
        # Get reviews 
        comp_revs = get_reviews(gd_link[:-4], 1, 25)
        
    except: # 2-25 did not work 
        
        # Print number error
        print("   Company " + company + " did not work, please check")
        
        # Go to next company
        continue
    
    
    # Try with next 25 pages
    try: # Pages 26-50
        
        # Print state
        print("   Getting reviews 26-50")
        
        # Get reviews
        comp_revs = pd.concat([comp_revs, get_reviews(gd_link[:-4], 26, 50)])
        
    except: # 26-50 did not work (but previous ones did)
        
        # Print number error
        print("   Company " + company + " only has " + str(len(comp_revs)) + " reviews, please check")
        
        # Save reviews
        comp_revs.to_csv("./01_company_reviews/" + company + "_reviews.csv", index=False)
        
        # Go to next company
        continue
    
    
    # Try with next 25 pages
    try: # Pages 51-75
        
        # Print state
        print("   Getting reviews 51-75")
        
        # Get reviews
        comp_revs = pd.concat([comp_revs, get_reviews(gd_link[:-4], 51, 75)])
        
    except: # 51-75 did not work (but previous ones did)
        
        # Print number error
        print("   Company " + company + " only has " + str(len(comp_revs)) + " reviews, please check")
        
        # Save reviews
        comp_revs.to_csv("./01_company_reviews/" + company + "_reviews.csv", index=False)
        
        # Go to next company
        continue
        
        
    # Try with next 25 pages
    try: # Pages 76-100
        
        # Print state
        print("   Getting reviews 76-100")
        
        # Get reviews
        comp_revs = pd.concat([comp_revs, get_reviews(gd_link[:-4], 76, 100)])
        
    except: # 76-100 did not work (but previous ones did)
        
        # Print number error
        print("   Company " + company + " only has " + str(len(comp_revs)) + " reviews, please check")
        
        # Save reviews
        comp_revs.to_csv("./01_company_reviews/" + company + "_reviews.csv", index=False)
        
        # Go to next company
        continue


    # Try with next 25 pages
    try: # Pages 100-125
        
        # Print state
        print("   Getting reviews 100-125")
        
        # Get reviews
        comp_revs = pd.concat([comp_revs, get_reviews(gd_link[:-4], 101, 125)])
        
        # Save reviews
        comp_revs.to_csv("./01_company_reviews/" + company + "_reviews.csv", index=False)
        
        # Print success
        print("   Company " + company + " finished successfully with " + str(len(comp_revs)) + " reviews :) ")
        
    except: # 100-125 did not work (but previous ones did)
        
        # Print number error
        print("   Company " + company + " only has " + str(len(comp_revs)) + " reviews, please check")
        
        # Save reviews
        comp_revs.to_csv("./01_company_reviews/" + company + "_reviews.csv", index=False)


The current company is: ARE
   Getting reviews 2-25
   Getting reviews 26-50
   Getting reviews 51-75
   Getting reviews 76-100
   Getting reviews 100-125
   Company ARE finished successfully with 51 reviews :) 

The current company is: BKNG
   Getting reviews 2-25
   Getting reviews 26-50
   Getting reviews 51-75
   Getting reviews 76-100
   Getting reviews 100-125
   Company BKNG finished successfully with 19 reviews :) 

The current company is: BXP
   Getting reviews 2-25
   Getting reviews 26-50
   Getting reviews 51-75
   Getting reviews 76-100
   Getting reviews 100-125
   Company BXP finished successfully with 131 reviews :) 

The current company is: CMS
   Getting reviews 2-25
   Getting reviews 26-50
   Getting reviews 51-75
   Getting reviews 76-100
   Getting reviews 100-125
   Company CMS finished successfully with 16 reviews :) 

The current company is: COO
   Getting reviews 2-25
   Getting reviews 26-50
   Getting reviews 51-75
   Getting reviews 76-100
   Getting revie